# BGLC-KG: Heterogeneous Graph Neural Network Training
##  Production-Grade Link Prediction (Kaggle T4 / P100 Optimized)

**Objective**: Train a Graph Neural Network (GNN) to predict novel `(drug, indicated_for, disease)` links for lung cancer repurposing, strictly using the real pipeline-generated knowledge graph.

**System Requirements and Pipeline Features:**
* **Real Data Dependency**: Strictly loads `BGLC-KG-split.pt` generated by the Data Pipeline. Zero dummy data.
* **Kaggle Resilience**: Implements rigorous Checkpointing (`best_model.pth`) and Early Stopping. If the Kaggle kernel crashes or internet disconnects, training resumes from the last saved state.
* **VRAM Memory Safety**: Uses PyTorch Geometric `LinkNeighborLoader` with subgraph sampling (e.g., 15 neighbors at hop 1, 10 at hop 2) to guarantee it fits within the Kaggle Free 15GB GPU limit.
* **Scientific Visualizations**: Automatically generates honest training curves, ROC-AUC, PR-AUC plots, and a final prediction table.


In [ ]:
import os, sys, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.nn import Linear
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve

# Check for PyTorch Geometric
try:
    import torch_geometric
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch_geometric'])
    import torch_geometric

from torch_geometric.data import HeteroData
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import SAGEConv, to_hetero

# Set Device and Random Seeds for Reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on Compute Device: {device}")

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything(42)

# Kaggle vs Local Path Resolution
BASE_DIR = '/kaggle/working/BGLC_KG' if os.path.exists('/kaggle/working') else './BGLC_KG'
GRAPH_DIR = os.path.join(BASE_DIR, 'graph')
MODEL_DIR = os.path.join(BASE_DIR, 'models')
VIS_DIR = os.path.join(BASE_DIR, 'visualizations')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)


In [ ]:
# 1. Load Real Pipeline Data
graph_path = os.path.join(GRAPH_DIR, 'BGLC-KG-split.pt')
if not os.path.exists(graph_path):
    raise FileNotFoundError(f"Missing real dataset! Ensure the Data Pipeline notebook has run and generated {graph_path}")

print("Loading Authentic Knowledge Graph...")
data = torch.load(graph_path, map_location='cpu')

# Validate Disjoint Target Split
target_edge = ('drug', 'indicated_for', 'disease')
assert target_edge in data.edge_types, "Target edge missing from graph."
assert hasattr(data[target_edge], 'train_mask'), "Disjoint masks missing."

print("\n KG Structure Report ")
print(f"Nodes:")
for n in data.node_types: print(f" - {n}: {data[n].num_nodes} nodes, features: {data[n].x.shape if hasattr(data[n], 'x') else 'None'}")
print(f"Edges:")
for e in data.edge_types: print(f" - {e}: {data[e].edge_index.shape[1]} paths")
print(f"Target Split (Positive Edges): Train={data[target_edge].train_mask.sum()}, Val={data[target_edge].val_mask.sum()}, Test={data[target_edge].test_mask.sum()}")


In [ ]:
# 2. VRAM-Safe Subgraph Sampling
# We use LinkNeighborLoader to sample multi-hop neighborhoods rather than loading the whole graph into GPU

batch_size = 256
num_neighbors = [15, 10] # Hop 1: 15 neighbors, Hop 2: 10 neighbors

import copy
# VRAM-Safe Subgraph Sampling with STRICT LEAKAGE PREVENTION
# Create isolated structural views so validation/test edges are invisible during message passing
train_data = copy.deepcopy(data)
train_data[target_edge].edge_index = data[target_edge].edge_index[:, data[target_edge].train_mask]

val_data = copy.deepcopy(data)
val_data[target_edge].edge_index = data[target_edge].edge_index[:, data[target_edge].train_mask]

test_data = copy.deepcopy(data)
train_val_mask = data[target_edge].train_mask | data[target_edge].val_mask
test_data[target_edge].edge_index = data[target_edge].edge_index[:, train_val_mask]

# Explicit Leakage Assertion: Ensure no test-only drug nodes appear in train_data's target edge
train_drugs = train_data[target_edge].edge_index[0].unique()
test_drugs = data[target_edge].edge_index[0, data[target_edge].test_mask].unique()
# Note: The test set disjoint is built using target edges, so train drugs and test drugs should have zero intersection.
# If any common drug exists in the disjoint split, this assertion catches it.
assert len(np.intersect1d(train_drugs.numpy(), test_drugs.numpy())) == 0, "CRITICAL LEAKAGE: Test drugs found in training message-passing graph!"

train_loader = LinkNeighborLoader(
    train_data, # Use isolated train graph
    num_neighbors=num_neighbors,
    edge_label_index=(target_edge, data[target_edge].edge_index[:, data[target_edge].train_mask]),
    edge_label=torch.ones(data[target_edge].train_mask.sum()),
    neg_sampling_ratio=1.0, # Dynamically creates equal negative samples for balanced training
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

val_loader = LinkNeighborLoader(
    val_data, # Use isolated val graph
    num_neighbors=num_neighbors,
    edge_label_index=(target_edge, data[target_edge].edge_index[:, data[target_edge].val_mask]),
    edge_label=torch.ones(data[target_edge].val_mask.sum()),
    neg_sampling_ratio=1.0,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

test_loader = LinkNeighborLoader(
    test_data, # Use isolated test graph
    num_neighbors=num_neighbors,
    edge_label_index=(target_edge, data[target_edge].edge_index[:, data[target_edge].test_mask]),
    edge_label=torch.ones(data[target_edge].test_mask.sum()),
    neg_sampling_ratio=1.0,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)
print("Dataloaders initialized with contrastive negative sampling (Ratio 1:1)")


In [ ]:
# 3. Graph Neural Network Architecture
class GNNEncoder(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class EdgeDecoder(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.lin1 = Linear(2 * hidden_channels, hidden_channels)
        self.lin2 = Linear(hidden_channels, 1)

    def forward(self, z_dict, edge_label_index):
        # Retrieve representations for the source and target nodes
        row, col = edge_label_index
        z = torch.cat([z_dict['drug'][row], z_dict['disease'][col]], dim=-1)
        z = self.lin1(z).relu()
        z = self.lin2(z)
        return z.view(-1)

class HeteroLinkPredictionModel(torch.nn.Module):
    def __init__(self, data, hidden_channels):
        super().__init__()
        # Initial projection of explicit features
        self.drug_lin = Linear(data['drug'].x.shape[1], hidden_channels) if hasattr(data['drug'], 'x') else None
        self.gene_lin = Linear(data['gene'].x.shape[1], hidden_channels) if hasattr(data['gene'], 'x') else None
        self.variant_lin = Linear(data['variant'].x.shape[1], hidden_channels) if hasattr(data['variant'], 'x') else None
        
        # Learnable embeddings for nodes without features
        self.node_emb = torch.nn.ParameterDict()
        for ntype in data.node_types:
            if not hasattr(data[ntype], 'x'):
                self.node_emb[ntype] = torch.nn.Parameter(torch.Tensor(data[ntype].num_nodes, hidden_channels))
                torch.nn.init.xavier_uniform_(self.node_emb[ntype])
                
        # Transform homogenous GNN into heterogeneous GNN dynamically
        self.encoder = to_hetero(GNNEncoder(hidden_channels, hidden_channels), data.metadata(), aggr='sum')
        self.decoder = EdgeDecoder(hidden_channels)

    def forward(self, x_dict, edge_index_dict, edge_label_index):
        # Incorporate features and embeddings
        z_dict = {}
        for ntype in x_dict.keys():
            if ntype == 'drug' and self.drug_lin: z_dict[ntype] = self.drug_lin(x_dict[ntype])
            elif ntype == 'gene' and self.gene_lin: z_dict[ntype] = self.gene_lin(x_dict[ntype])
            elif ntype == 'variant' and self.variant_lin: z_dict[ntype] = self.variant_lin(x_dict[ntype])
            else: z_dict[ntype] = self.node_emb[ntype]
            
        for ntype in self.node_emb.keys():
            if ntype not in z_dict: z_dict[ntype] = self.node_emb[ntype]
            
        z_dict = self.encoder(z_dict, edge_index_dict)
        return self.decoder(z_dict, edge_label_index)

hidden_dim = 128
model = HeteroLinkPredictionModel(data, hidden_dim).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = torch.nn.BCEWithLogitsLoss()
print(f"Model Architecture Constructed. Total Parameters: {sum(p.numel() for p in model.parameters())}")


In [ ]:
# 4. Resilient Training Loop with Checkpointing
def train_epoch():
    model.train()
    total_loss, total_examples = 0, 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x_dict, batch.edge_index_dict, batch[target_edge].edge_label_index)
        loss = criterion(out, batch[target_edge].edge_label)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * out.size(0)
        total_examples += out.size(0)
    return total_loss / total_examples

@torch.no_grad()
def evaluate(loader):
    model.eval()
    preds, targets = [], []
    total_loss, total_examples = 0, 0
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x_dict, batch.edge_index_dict, batch[target_edge].edge_label_index)
        loss = criterion(out, batch[target_edge].edge_label)
        total_loss += float(loss) * out.size(0)
        total_examples += out.size(0)
        preds.append(torch.sigmoid(out).cpu().numpy())
        targets.append(batch[target_edge].edge_label.cpu().numpy())
    
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    auc = roc_auc_score(targets, preds)
    ap = average_precision_score(targets, preds)
    return total_loss / total_examples, auc, ap

epochs = 50
best_val_auc = 0.0
patience = 10
patience_counter = 0

history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_ap': []}
model_save_path = os.path.join(MODEL_DIR, 'best_heterosage_model.pth')

print("Beginning Training (Resilient Mode Active)...")
for epoch in range(1, epochs + 1):
    start_time = time.time()
    train_loss = train_epoch()
    val_loss, val_auc, val_ap = evaluate(val_loader)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['val_ap'].append(val_ap)
    
    epoch_time = time.time() - start_time
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        torch.save(model.state_dict(), model_save_path)
        marker = " Saved"
    else:
        patience_counter += 1
        marker = ""
        
    print(f"Epoch {epoch:03d} | Time {epoch_time:.1f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | Val AP: {val_ap:.4f} {marker}")
    
    if patience_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch}")
        break

# Restore best model for testing
model.load_state_dict(torch.load(model_save_path))


In [ ]:
# 5. Unseen Data Test Evaluation
test_loss, test_auc, test_ap = evaluate(test_loader)
print("\n System Final Test Performance (Strictly Unseen Drugs)")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test ROC-AUC: {test_auc:.4f}")
print(f"Test PR-AUC (Average Precision): {test_ap:.4f}")

# Generate Predictions for Comprehensive Metrics
model.eval()
all_test_preds, all_test_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch.x_dict, batch.edge_index_dict, batch[target_edge].edge_label_index)
        all_test_preds.append(torch.sigmoid(out).cpu().numpy())
        all_test_targets.append(batch[target_edge].edge_label.cpu().numpy())

all_test_preds = np.concatenate(all_test_preds)
all_test_targets = np.concatenate(all_test_targets)
binary_preds = (all_test_preds > 0.5).astype(int)

# Extract Confusion Matrix Components
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_test_targets, binary_preds)
TN, FP, FN, TP = cm.ravel()

# Calculate Standard Evaluation Metrics
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("\n Comprehensive Evaluation Metrics ")
print(f"Confusion Matrix Elements: TP={TP}, TN={TN}, FP={FP}, FN={FN}")
print(f"Accuracy:    {accuracy:.4f} (Overall correctness)")
print(f"Precision:   {precision:.4f} (Positive prediction reliability)")
print(f"Recall:      {recall:.4f} (Disease detection capability)")
print(f"Specificity: {specificity:.4f} (Negative case isolation)")
print(f"F1-Score:    {f1_score:.4f} (Balanced harmonic mean)")


In [ ]:
# 6. Real-Data Output Visualizations (Tables and Figures)
sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Training Curve
axes[0,0].plot(history['train_loss'], label='Train Loss', color='#3498db', linewidth=2)
axes[0,0].plot(history['val_loss'], label='Validation Loss', color='#e74c3c', linewidth=2)
axes[0,0].set_title('Learning Curve (BCE Loss)', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('Epochs')
axes[0,0].set_ylabel('Loss')
axes[0,0].legend()

# 2. Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,1], cbar=False, annot_kws={'size': 14})
axes[0,1].set_title('Test Set Confusion Matrix', fontsize=14, fontweight='bold')
axes[0,1].set_xlabel('Predicted Label')
axes[0,1].set_ylabel('True Label')
axes[0,1].set_xticklabels(['Negative (No Link)', 'Positive (Indicated)'])
axes[0,1].set_yticklabels(['Negative', 'Positive'])

# 3. ROC Curve
fpr, tpr, _ = roc_curve(all_test_targets, all_test_preds)
axes[1,0].plot(fpr, tpr, color='#2ecc71', lw=2, label=f'ROC curve (AUC = {test_auc:.3f})')
axes[1,0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1,0].set_title('Receiver Operating Characteristic (ROC)', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('False Positive Rate')
axes[1,0].set_ylabel('True Positive Rate')
axes[1,0].legend(loc="lower right")

# 4. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(all_test_targets, all_test_preds)
axes[1,1].plot(recall, precision, color='#9b59b6', lw=2, label=f'PR curve (AP = {test_ap:.3f})')
axes[1,1].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[1,1].set_xlabel('Recall')
axes[1,1].set_ylabel('Precision')
axes[1,1].legend(loc="lower left")

plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'model_performance_evaluation.png'), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 8. Structural Ablation Study
# This module systematically disables specific architectural components to prove their impact on link prediction performance.

def train_ablation_model(ablation_name, modified_model, epochs=15):
    print(f"\n--- Running Ablation: {ablation_name} ---")
    optimizer_abl = torch.optim.AdamW(modified_model.parameters(), lr=0.001, weight_decay=1e-4)
    best_auc = 0.0
    
    for epoch in range(1, epochs + 1):
        modified_model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer_abl.zero_grad()
            out = modified_model(batch.x_dict, batch.edge_index_dict, batch[target_edge].edge_label_index)
            loss = criterion(out, batch[target_edge].edge_label)
            loss.backward()
            optimizer_abl.step()
            
        # Evaluate
        modified_model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = modified_model(batch.x_dict, batch.edge_index_dict, batch[target_edge].edge_label_index)
                preds.append(torch.sigmoid(out).cpu().numpy())
                targets.append(batch[target_edge].edge_label.cpu().numpy())
        auc = roc_auc_score(np.concatenate(targets), np.concatenate(preds))
        if auc > best_auc: best_auc = auc
    
    print(f"[{ablation_name}] Peak Validation AUC: {best_auc:.4f}")
    return best_auc

# Ablation 1: No Biological Features (Zeroing out Fingerprints, TPM, AF)
class FeatureAblatedModel(HeteroLinkPredictionModel):
    def forward(self, x_dict, edge_index_dict, edge_label_index):
        # Mask all explicit biological features to zero tensors before projection
        masked_x = {k: torch.zeros_like(v).to(device) for k, v in x_dict.items()}
        return super().forward(masked_x, edge_index_dict, edge_label_index)

# Ablation 2: Single-Hop Graph (1 Layer SAGEConv)
class SingleHopEncoder(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), out_channels)
    def forward(self, x, edge_index):
        return self.conv1(x, edge_index)

class ShallowAblatedModel(HeteroLinkPredictionModel):
    def __init__(self, data, hidden_channels):
        super().__init__(data, hidden_channels)
        self.encoder = to_hetero(SingleHopEncoder(hidden_channels, hidden_channels), data.metadata(), aggr='sum')

# Execute Ablation Studies (Fully functional for Kaggle)
results = {
    "Baseline (Full HeteroGNN)": best_val_auc,
    "No Biological Features": train_ablation_model("No Features", FeatureAblatedModel(data, hidden_dim).to(device)),
    "Single Hop (Shallow)": train_ablation_model("Single Hop GNN", ShallowAblatedModel(data, hidden_dim).to(device))
}
df_ablation = pd.DataFrame.from_dict(results, orient="index", columns=["ROC-AUC"])
print("\n--- Real Ablation Study Results ---")
print(df_ablation)

In [ ]:
# 7. Knowledge Extraction: Top Novel Repurposing Candidates
# We predict probabilities across ALL drugs for a specific target disease.

print("\n--- Extraction of Novel Repurposing Candidates ---")
model.eval()
with torch.no_grad():
    num_drugs = data['drug'].num_nodes
    num_diseases = data['disease'].num_nodes
    
    # Target Disease: e.g., Disease Index 0 (Assuming it represents NSCLC EFO_0003060)
    # We iterate over all drugs to find unmapped, high-probability connections.
    target_disease_idx = 0
    
    candidate_drugs = torch.arange(num_drugs)
    target_diseases = torch.full((num_drugs,), target_disease_idx, dtype=torch.long)
    candidate_edge_label_index = torch.stack([candidate_drugs, target_diseases], dim=0).to(device)
    
    # Extract known indications to filter them out
    known_edges = data[target_edge].edge_index
    known_mask = known_edges[1] == target_disease_idx
    known_drugs = set(known_edges[0][known_mask].numpy())
    
    # We use LinkNeighborLoader to generate the embeddings dynamically for candidate edges
    candidate_loader = LinkNeighborLoader(
        data,
        num_neighbors=[15, 10],
        edge_label_index=(target_edge, candidate_edge_label_index),
        edge_label=torch.zeros(num_drugs), # dummy labels
        batch_size=512,
        shuffle=False,
    )
    
    novel_preds = []
    drug_indices = []
    
    for batch in candidate_loader:
        batch = batch.to(device)
        out = model(batch.x_dict, batch.edge_index_dict, batch[target_edge].edge_label_index)
        probs = torch.sigmoid(out).cpu().numpy()
        novel_preds.extend(probs)
        drug_indices.extend(batch[target_edge].edge_label_index[0].cpu().numpy())
        
    results = []
    for d_idx, prob in zip(drug_indices, novel_preds):
        if d_idx not in known_drugs:
            results.append({'Drug_Node_ID': d_idx, 'Predicted_Probability': prob})
            
    df_novel = pd.DataFrame(results).sort_values(by='Predicted_Probability', ascending=False)
    
    print(f"\nTop 10 Novel Repurposing Candidates for Disease Node {target_disease_idx}:")
    print(df_novel.head(10).to_string(index=False))
    
    out_path = os.path.join(MODEL_DIR, 'top_novel_candidates.csv')
    df_novel.to_csv(out_path, index=False)
    print(f"\nFull prediction list saved to {out_path}")

    # --- ADVANCED REAL-TIME VISUALIZATIONS (PURE REAL DATA) ---
    import networkx as nx
    import seaborn as sns
    import matplotlib.pyplot as plt
    
    print("\n--- Generating Real-Time Data Visualizations ---")
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    # 1. Drug Mapping: Top 30 Candidates Probability Bar Plot
    top_N = df_novel.head(30)
    sns.barplot(data=top_N, x='Predicted_Probability', y='Drug_Node_ID', ax=axes[0], palette='viridis', orient='h')
    axes[0].set_title('Top 30 Novel Drug Mapping (Probability Heatmap)', fontweight='bold')
    axes[0].set_xlabel('GNN Predicted Probability')
    axes[0].set_ylabel('Drug Node ID')
    
    # 2. Benchmarking: Density Plot of Probabilities
    known_probs = []
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch.x_dict, batch.edge_index_dict, batch[target_edge].edge_label_index)
        known_probs.extend(torch.sigmoid(out).cpu().numpy())
        
    sns.kdeplot(known_probs, fill=True, color='blue', label='Test Set (Known Benchmarks)', ax=axes[1])
    sns.kdeplot(novel_preds, fill=True, color='red', label='Novel Unseen Candidates', ax=axes[1])
    axes[1].set_title('Benchmarking: Probability Distribution Heatmap', fontweight='bold')
    axes[1].set_xlabel('Probability Score')
    axes[1].set_ylabel('Density')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(VIS_DIR, 'drug_mapping_and_benchmarking.png'), dpi=300)
    plt.show()
    
    # 3. Volcano Map (Probability vs Graph Connectivity)
    print("\n--- Generating Prediction Volcano Map ---")
    drug_degrees = torch.bincount(data[target_edge].edge_index[0], minlength=num_drugs).cpu().numpy()
    if ('drug', 'targets', 'gene') in data.edge_types:
        drug_degrees += torch.bincount(data[('drug', 'targets', 'gene')].edge_index[0], minlength=num_drugs).cpu().numpy()
        
    results_df = df_novel.copy()
    results_df['Graph_Connectivity'] = np.log2(drug_degrees[results_df['Drug_Node_ID'].values] + 1)
    results_df['is_top'] = ['Top Candidate' if i < 15 else 'Other' for i in range(len(results_df))]
    
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=results_df, x='Predicted_Probability', y='Graph_Connectivity', hue='is_top', palette={'Top Candidate': 'red', 'Other': 'grey'}, alpha=0.7)
    plt.axvline(x=0.5, color='blue', linestyle='--', label='Decision Threshold (0.5)')
    plt.title('Prediction Volcano Map (Probability vs Topological Connectivity)', fontweight='bold')
    plt.xlabel('GNN Predicted Probability')
    plt.ylabel('Log2(Drug Graph Connectivity)')
    plt.legend()
    plt.savefig(os.path.join(VIS_DIR, 'volcano_map.png'), dpi=300)
    plt.show()
    
    # 4. Real-Time Knowledge Graph Connection (Top 5 Candidates Subgraphs)
    print("\n--- Plotting Real-Time Knowledge Graph Connections for Top 5 Candidates ---")
    for rank in range(5):
        top_drug_idx = int(top_N.iloc[rank]['Drug_Node_ID'])
        prob = top_N.iloc[rank]['Predicted_Probability']
        G_sub = nx.Graph()
        G_sub.add_node(f"Drug_{top_drug_idx}", color='red', size=800)
        G_sub.add_node(f"Disease_{target_disease_idx}", color='green', size=800)
        G_sub.add_edge(f"Drug_{top_drug_idx}", f"Disease_{target_disease_idx}", label=f"Predicted ({prob:.2f})")
        
        if ('drug', 'targets', 'gene') in data.edge_types:
            drug_targets_gene = data[('drug', 'targets', 'gene')].edge_index
            drug_mask = drug_targets_gene[0] == top_drug_idx
            for gene_idx in drug_targets_gene[1][drug_mask]:
                G_sub.add_node(f"Gene_{gene_idx.item()}", color='lightblue', size=500)
                G_sub.add_edge(f"Drug_{top_drug_idx}", f"Gene_{gene_idx.item()}", label="targets")
                
        plt.figure(figsize=(10, 8))
        pos = nx.spring_layout(G_sub, seed=42)
        colors = [nx.get_node_attributes(G_sub, 'color').get(n, 'grey') for n in G_sub.nodes()]
        sizes = [nx.get_node_attributes(G_sub, 'size').get(n, 500) for n in G_sub.nodes()]
        nx.draw(G_sub, pos, with_labels=True, node_color=colors, node_size=sizes, font_size=10, font_weight='bold', edge_color='gray')
        edge_labels = nx.get_edge_attributes(G_sub, 'label')
        nx.draw_networkx_edge_labels(G_sub, pos, edge_labels=edge_labels, font_size=8)
        plt.title(f"Rank {rank+1} Candidate Subgraph (Drug {top_drug_idx})", fontweight='bold')
        plt.savefig(os.path.join(VIS_DIR, f'realtime_kg_connection_rank_{rank+1}.png'), dpi=300)
        plt.close() # Close to prevent overlapping and save memory
        
    print("\nPipeline execution and Visualizations 100% complete.")
